# Algorithm Tutorial - Interactive Examples

Welcome to this interactive tutorial! This notebook demonstrates how to run our algorithms with practical examples.

## 📚 Table of Contents

- [1. Datasets and Query Applicability](#1.-Datasets-and-Query-Applicability)
- [2. Unified Algorithm Interface](#2.-Unified-Algorithm-Interface)
- [3. Example Usage](#3.-Example-Usage)
  - [3.1 Quick Reference](#3.1-Quick-Reference)
  - [3.2 Example Queries](#3.2-Example-Queries)

## 🎯 Overview

This tutorial covers:

1. **Dataset Information** - Understanding the provided datasets and their join relations
2. **Query Reference** - Query specifications with LEX and SUM orderings
3. **Algorithm Interface** - Using the unified `AccessAlgorithm` class
4. **Practical Examples** - Step-by-step examples for:
   - 🔤 **LEX Order Queries** (LEX_1 to LEX_13)
   - 🧮 **SUM Order Queries** (SUM_1 to SUM_7)
5. **Direct vs Single Access** - Understanding when to use each approach

### Quick Navigation to Queries

**LEX Order Examples:**
- [Query LEX_1](#Query-LEX_1) - Basic join with full order
- [Query LEX_2](#Query-LEX_2) - Partial order (Single Access only)
- [Query LEX_3](#Query-LEX_3) - Cartesian product
- [Query LEX_4](#Query-LEX_4) - Cartesian with partial order
- [Query LEX_5](#Query-LEX_5) - Path query (3 tables)
- [Query LEX_6](#Query-LEX_6) - Path with disruptive trio
- [Query LEX_7](#Query-LEX_7) - Path with minimal order
- [Query LEX_8](#Query-LEX_8) - Star query (4 tables)
- [Query LEX_9](#Query-LEX_9) - Star with projection
- [Query LEX_10](#Query-LEX_10) - Star with partial order
- [Query LEX_11](#Query-LEX_11) - Complex schema (multi-attribute)
- [Query LEX_12](#Query-LEX_12) - Complex with special order
- [Query LEX_13](#Query-LEX_13) - Complex with projection

**SUM Order Examples:**
- [Query SUM_1](#Query-SUM_1) - Sum over all variables
- [Query SUM_2](#Query-SUM_2) - Sum over subset
- [Query SUM_3](#Query-SUM_3) - Weighted sum (Cartesian)
- [Query SUM_4](#Query-SUM_4) - Sum on path query
- [Query SUM_5](#Query-SUM_5) - Non-tractable sum
- [Query SUM_6](#Query-SUM_6) - Sum on star query
- [Query SUM_7](#Query-SUM_7) - Complex sum order

## 1. Datasets and Query Applicability 

We provide 5 datasets for examples (under folder `datasets\`). Because each algorithm has their specific applicability, we summary the information of datasets and example queries for reference. 

You can also build your own tables and join relations 

| Dataset ID | Number of Tables | Join Relation                                     | 
|------------|------------------|---------------------------------------------------|
| 1          | 2                | $R(a, b) \Join_b S(b, c)$                         |
| 2          | 2                | $R(a, b) \times S(c, d)$                          | 
| 3          | 3                | $R(a, b) \Join_b S(b, c) \Join_c T(c, d)$         |
| 4          | 4                | $R(a, b) \Join_b S(a, c) \Join_b T(a, d) \Join_b U(a, e)  $      | 
| 5          | 4                | $R(a, b, c) \Join_b S(a, d) \Join_b T(b, e) \Join_b U(c, f, g)  $        |


Query
| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access | Comment |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|---------|
| [LEX_1](#Query-LEX_1)    | 1          | 1. a <br> 2. b <br> 3. c                | a, b, c        | Y             | Y             | |
| [LEX_2](#Query-LEX_2)    | 1          | 1. a <br> 2. c                          | a, b, c        | N             | Y             | |
| [LEX_3](#Query-LEX_3)    | 2          | 1. a (descending)<br> 2. b <br> 3. c                | a, b, c        | Y             | Y             | |
| [LEX_4](#Query-LEX_4)    | 2          | 1. a <br> 2. c                          | a, b, c        | Y             | Y             | |
| [LEX_5](#Query-LEX_5)    | 3          | 1. b <br> 2. a <br> 3. c <br> 4. d      | a, b, c, d     | Y             | Y             | |
| [LEX_6](#Query-LEX_6)    | 3          | 1. a <br> 2. c <br> 3. b <br> 4. d      | a, b, c, d     | N             | Y             | |
| [LEX_7](#Query-LEX_7)    | 3          | 1. a <br> 2. d                          | a, b, c, d     | N             | Y             | |
| [LEX_8](#Query-LEX_8)    | 4          | 1. a <br> 2. b <br> 3. c <br> 4. d <br> 5. e     | a, b, c, d, e     | Y             | Y             | |
| [LEX_9](#Query-LEX_9)    | 4          | 1. b <br> 2. a <br> 3. c                | a, b, c, d, e     | Y             | Y             | |
| [LEX_10](#Query-LEX_10)    | 4          | 1. b <br> 2. c                 | a, b, c, d, e     | N             | Y             | |
| [LEX_11](#Query-LEX_11)    | 5          | 1. b <br> 2. a <br> 3. d <br> 4. e <br>     | a, b, c, d, e, f, g     | Y             | Y             | |
| [LEX_12](#Query-LEX_12)    | 5          | 1. f <br> 2. c <br> 3. d                 | a, b, c, d, e, f     | N             | Y             | |
| [LEX_13](#Query-LEX_13)    | 5          | 1. b <br> 2. e <br> 3. c                | a, b, c, e, g     | N             | Y             | |
| [SUM_1](#Query-SUM_1)    | 1          | a + b + c                 | a, b, c        | N             | Y             | |
| [SUM_2](#Query-SUM_2)    | 1          | a + b                     | a, b, c        | Y             | Y             | |
| [SUM_3](#Query-SUM_3)    | 2          | a + 2b + c                | a, b, c        | N             | Y             | |
| [SUM_4](#Query-SUM_4)    | 3          | a + b + c                 | a, b, c, d     | N             | Y             | |
| [SUM_5](#Query-SUM_5)    | 3          | a + d                     | a, b, c, d     | N             | N             | |
| [SUM_6](#Query-SUM_6)    | 4          | b + d       | a, b, c, d, e     | N             | Y             | |


## 2. Unified Algorithm Interface

We provide a unified `AccessAlgorithm` class that wraps all access algorithms:

### Algorithm Types

| Order Type | Access Type | Description |
|------------|-------------|-------------|
| LEX | Direct | Preprocesses data, allows multiple efficient accesses |
| LEX | Single | Accesses each position from scratch (no preprocessing) |
| SUM | Direct | Preprocesses data, allows multiple efficient accesses |
| SUM | Single | Accesses each position from scratch (no preprocessing) |

### Key Features
- **Direct Access**: One-time preprocessing, then efficient repeated access to any k-th result
- **Single Access**: No preprocessing needed, compute each result independently
- **LEX Order**: Lexicographic ordering of variables
- **SUM Order**: Ordering by weighted sum of variables

## 3. Example Usage

Let's demonstrate all four variants of the algorithm with practical examples.

### 3.1 Quick Reference


#### 🔤 Direct Access with LEX order

```python
algo = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables, 
    order_type='lex', 
    lex_order={'a': 1, 'b': -1}, # first sort by a (ascending), then sort by b (descending)
    access_type='direct'
)   # finish preprocess when initializing

# Get specific result
result = algo.get_k(5) # using built data structure 

# Get total count (Direct Access only)
total = algo.get_total_count()

# Get all results
all_results = algo.get_all()
```
---
#### 🔤 Single Access with LEX order

```python
algo = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables, 
    order_type='lex', 
    lex_order={'a': 1, 'c': -1}, # first sort by a (ascending), then sort by c (descending)
    access_type='single'
)   

# Get specific result
result = algo.get_k(5)
# Get all results
all_results = algo.get_all()
```
---
#### 🧮 Direct Access with SUM order

```python
algo = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum', 
    sum_order={'a': 1, 'c': -1}, # sort according to the values of a-c
    access_type='direct'
)   # finish preprocess when initializing

# Get specific result
result = algo.get_k(5) # using built data structure 

# Get total count (Direct Access only)
total = algo.get_total_count()

# Get all results
all_results = algo.get_all()
```

---

#### 🧮 Single Access with SUM order

```python
algo = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum', 
    sum_order={'a': 1, 'c': -1}, # sort according to the values of a-c
    access_type='single'
)   # no any preprocessing

# Get specific result
result = algo.get_k(10) 

# Get first N results
first_n = algo.get_all(limit=10)
```

---

### 3.2 Example Queries

First, let's import the necessary libraries:

In [ ]:
%xmode Minimal
# Import required libraries
import pandas as pd
from pathlib import Path

# Import tutorial utilities
from tutorial_utils import load_dataset
from pandasCompare import pandas_join_lex, pandas_join_sum
from access_algorithm import AccessAlgorithm

# Display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")

---

#### Query LEX_1

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_1](#Query-LEX_1)    | 1          | 1. a <br> 2. b <br> 3. c                | a, b, c        | Y             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('b', 'c'))
]
data = load_dataset("datasets/data_1", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c']
lex_order = {'a': 1, 'b': 1, 'c': 1}

# Query LEX_1: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_1: Direct Access with LEX order
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

print("=== Direct Access - LEX Order ===")
print(f"Total count: {algo_da_lex.get_total_count()}")
print(f"\nFirst result (k=0): {algo_da_lex.get_k(0)}")
print(f"Second result (k=1): {algo_da_lex.get_k(1)}")
print(f"Third result (k=2): {algo_da_lex.get_k(2)}")

print("\nAll results:")
all_results = algo_da_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

In [ ]:
# Query LEX_1: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")
print(f"Fifth result (k=4): {algo_sa_lex.get_k(4)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

---

#### Query LEX_2

 Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_2](#Query-LEX_2)    | 1          | 1. a <br> 2. c                          | a, b, c        | N             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('b', 'c'))
]
data = load_dataset("datasets/data_1", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c']
lex_order = {'a': 1, 'c': 1}

# Query LEX_2: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_2: Direct Access with LEX order - not available
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

In [ ]:
# Query LEX_2: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")
print(f"Fifth result (k=4): {algo_sa_lex.get_k(4)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

---

#### Query LEX_3

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_3](#Query-LEX_3)    | 2          | 1. a (descending)<br> 2. b <br> 3. c                | a, b, c        | Y             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('c', 'd'))
]
data = load_dataset("datasets/data_2", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c']
lex_order = {'a': -1, 'b': 1, 'c': 1}

# Query LEX_3: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_3: Direct Access with LEX order
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

print("=== Direct Access - LEX Order ===")
print(f"Total count: {algo_da_lex.get_total_count()}")
print(f"\nFirst result (k=0): {algo_da_lex.get_k(0)}")
print(f"Second result (k=1): {algo_da_lex.get_k(1)}")
print(f"Third result (k=2): {algo_da_lex.get_k(2)}")

print("\nAll results:")
all_results = algo_da_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

In [ ]:
# Query LEX_3: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")
print(f"Fifth result (k=4): {algo_sa_lex.get_k(4)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

---


#### Query LEX_4

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_4](#Query-LEX_4)    | 2          | 1. a <br> 2. c                          | a, b, c        | Y             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('c', 'd'))
]
data = load_dataset("datasets/data_2", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c']
lex_order = {'a': 1, 'c': 1}

# Query LEX_4: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_4: Direct Access with LEX order
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

print("=== Direct Access - LEX Order ===")
print(f"Total count: {algo_da_lex.get_total_count()}")
print(f"\nFirst result (k=0): {algo_da_lex.get_k(0)}")
print(f"Second result (k=1): {algo_da_lex.get_k(1)}")
print(f"Third result (k=2): {algo_da_lex.get_k(2)}")

print("\nAll results:")
all_results = algo_da_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

In [ ]:
# Query LEX_4: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")
print(f"Fifth result (k=4): {algo_sa_lex.get_k(4)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

---

#### Query LEX_5

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_5](#Query-LEX_5)    | 3          | 1. b <br> 2. a <br> 3. c <br> 4. d      | a, b, c, d     | Y             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('b', 'c')),
    ('T', ('c', 'd'))
]
data = load_dataset("datasets/data_3", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c', 'd']
lex_order = {'a': 1, 'b': 1, 'c': 1, 'd': 1}

# Query LEX_5: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_5: Direct Access with LEX order
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

print("=== Direct Access - LEX Order ===")
print(f"Total count: {algo_da_lex.get_total_count()}")
print(f"\nFirst result (k=0): {algo_da_lex.get_k(0)}")
print(f"Second result (k=1): {algo_da_lex.get_k(1)}")
print(f"Third result (k=2): {algo_da_lex.get_k(2)}")

print("\nAll results:")
all_results = algo_da_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

In [ ]:
# Query LEX_5: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")
print(f"Fifth result (k=4): {algo_sa_lex.get_k(4)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

---

#### Query LEX_6

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_6](#Query-LEX_6)    | 3          | 1. a <br> 2. c <br> 3. b <br> 4. d      | a, b, c, d     | N             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('b', 'c')),
    ('T', ('c', 'd'))
]
data = load_dataset("datasets/data_3", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c', 'd']
lex_order = {'a': 1, 'c': 1, 'b': 1, 'd': 1}

# Query LEX_6: show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_6: Direct Access with LEX order
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

In [ ]:
# Query LEX_6: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")
print(f"Fifth result (k=4): {algo_sa_lex.get_k(4)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

---

#### Query LEX_7

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_7](#Query-LEX_7)    | 3          | 1. a <br> 2. c <br> 3. b <br> 4. d      | a, b, c, d     | N             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('b', 'c')),
    ('T', ('c', 'd'))
]
data = load_dataset("datasets/data_3", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c','d']
lex_order = {'a': 1, 'd': 1}

# Query LEX_7: show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_7: Direct Access with LEX order
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

In [ ]:
# Query LEX_7: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")
print(f"Fifth result (k=4): {algo_sa_lex.get_k(4)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

---

#### Query LEX_8

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_8](#Query-LEX_8)    | 4          | 1. a <br> 2. b <br> 3. c <br> 4. d <br> 5. e     | a, b, c, d, e     | Y             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('a', 'c')),
    ('T', ('a', 'd')),
    ('U', ('a', 'e')),
]
data = load_dataset("datasets/data_4", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a', 'b', 'c', 'd', 'e']
lex_order = {'a': 1, 'b': 1, 'c': 1, 'd': 1, 'e': 1}

# Query LEX_8: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_8: Direct Access with LEX order
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

print("=== Direct Access - LEX Order ===")
print(f"Total count: {algo_da_lex.get_total_count()}")
print(f"\nFirst result (k=0): {algo_da_lex.get_k(0)}")
print(f"Second result (k=1): {algo_da_lex.get_k(1)}")
print(f"Third result (k=2): {algo_da_lex.get_k(2)}")

print("\nAll results:")
all_results = algo_da_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

In [ ]:
# Query LEX_8: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")
print(f"Fifth result (k=4): {algo_sa_lex.get_k(4)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

---

#### Query LEX_9

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_9](#Query-LEX_9)    | 4          | 1. b <br> 2. a <br> 3. c                | a, b, c, d, e     | Y             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('a', 'c')),
    ('T', ('a', 'd')),
    ('U', ('a', 'e')),
]
data = load_dataset("datasets/data_4", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a', 'b', 'c', 'd', 'e']
lex_order = {'b': 1, 'a': 1, 'c': 1}

# Query LEX_9: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_9: Direct Access with LEX order
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

print("=== Direct Access - LEX Order ===")
print(f"Total count: {algo_da_lex.get_total_count()}")
print(f"\nFirst result (k=0): {algo_da_lex.get_k(0)}")
print(f"Second result (k=1): {algo_da_lex.get_k(1)}")
print(f"Third result (k=2): {algo_da_lex.get_k(2)}")

print("\nAll results:")
all_results = algo_da_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

In [ ]:
# Query LEX_9: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")
print(f"Fifth result (k=4): {algo_sa_lex.get_k(4)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

---

#### Query LEX_10

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_10](#Query-LEX_10)    | 4          | 1. b <br> 2. c                 | a, b, c, d, e     | N             | Y             | |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('a', 'c')),
    ('T', ('a', 'd')),
    ('U', ('a', 'e')),
]
data = load_dataset("datasets/data_4", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a', 'b', 'c', 'd','e']
lex_order = {'b': 1, 'c': 1}

# Query LEX_10: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_10: Direct Access with LEX order
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

print("=== Direct Access - LEX Order ===")
print(f"Total count: {algo_da_lex.get_total_count()}")
print(f"\nFirst result (k=0): {algo_da_lex.get_k(0)}")
print(f"Second result (k=1): {algo_da_lex.get_k(1)}")
print(f"Third result (k=2): {algo_da_lex.get_k(2)}")

print("\nAll results:")
all_results = algo_da_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

In [ ]:
# Query LEX_10: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")
print(f"Fifth result (k=4): {algo_sa_lex.get_k(4)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

--- 

#### Query LEX_11

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_11](#Query-LEX_11)    | 5          | 1. b <br> 2. a <br> 3. d <br> 4. e <br>     | a, b, c, d, e, f, g     | Y             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b', 'c')),
    ('S', ('a', 'd')),
    ('T', ('b', 'e')),
    ('U', ('c', 'f', 'g')),
]
data = load_dataset("datasets/data_5", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a', 'b', 'c', 'd','e', 'f', 'g']
lex_order = {'b': 1, 'a': 1, 'd': 1, 'e': 1}

# Query LEX_11: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_11: Direct Access with LEX order
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

print("=== Direct Access - LEX Order ===")
print(f"Total count: {algo_da_lex.get_total_count()}")
print(f"\nFirst result (k=0): {algo_da_lex.get_k(0)}")
print(f"Second result (k=1): {algo_da_lex.get_k(1)}")
print(f"Third result (k=2): {algo_da_lex.get_k(2)}")

print("\nAll results:")
all_results = algo_da_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

In [ ]:
# Query LEX_11: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

---

#### Query LEX_12

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_12](#Query-LEX_12)    | 5          | 1. f <br> 2. c <br> 3. d                 | a, b, c, d, e, f     | N             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b', 'c')),
    ('S', ('a', 'd')),
    ('T', ('b', 'e')),
    ('U', ('c', 'f', 'g')),
]
data = load_dataset("datasets/data_5", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a', 'b', 'c', 'd','e', 'f']
lex_order = {'f': 1, 'c': 1, 'd': 1}

# Query LEX_12: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_12: Direct Access with LEX order
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

In [ ]:
# Query LEX_12: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

---

#### Query LEX_13

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [LEX_13](#Query-LEX_13)    | 5          | 1. b <br> 2. e   <br> 3. c              | a, b, c, e, g     | N             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b', 'c')),
    ('S', ('a', 'd')),
    ('T', ('b', 'e')),
    ('U', ('c', 'f', 'g')),
]
data = load_dataset("datasets/data_5", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a', 'b', 'c', 'e', 'g']
lex_order = {'b': 1, 'e': 1, 'c': 1}

# Query LEX_13: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_lex(atoms, data, lex_order, free_variables, output_variables=None)

In [ ]:
# Query LEX_13: Direct Access with LEX order
algo_da_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='direct'
)

print("=== Direct Access - LEX Order ===")
print(f"Total count: {algo_da_lex.get_total_count()}")
print(f"\nFirst result (k=0): {algo_da_lex.get_k(0)}")
print(f"Second result (k=1): {algo_da_lex.get_k(1)}")
print(f"Third result (k=2): {algo_da_lex.get_k(2)}")

print("\nAll results:")
all_results = algo_da_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)


In [ ]:
# Query LEX_13: Single Access with LEX order
algo_sa_lex = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='lex',
    lex_order=lex_order,
    access_type='single'
)

print("=== Single Access - LEX Order ===")
print(f"First result (k=0): {algo_sa_lex.get_k(0)}")

print("\nAll results:")
all_results = algo_sa_lex.get_all()
pd.DataFrame(all_results, columns=free_variables)

---

#### Query SUM_1

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [SUM_1](#Query-SUM_1)    | 1          | a + b + c                 | a, b, c        | N             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('b', 'c'))
]
data = load_dataset("datasets/data_1", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c']
sum_order = {'a': 1, 'b': 1, 'c': 1}

# Query SUM_1: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_sum(atoms, data, sum_order, free_variables, output_variables=None)

In [ ]:
# Query SUM_1: Direct Access with SUM order
algo_da_sum = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum', 
    sum_order=sum_order, 
    access_type='direct'
)

print("=== Direct Access - SUM Order ===")
print(f"Total count: {algo_da_sum.get_total_count()}")
print(f"\nFirst result (k=0): {algo_da_sum.get_k(0)}")
print(f"Second result (k=1): {algo_da_sum.get_k(1)}")
print(f"Third result (k=2): {algo_da_sum.get_k(2)}")

print("\nAll results:")
all_results = algo_da_sum.get_all()
pd.DataFrame(all_results, columns=free_variables)

In [ ]:
# Query SUM_1: Single Access with SUM order
algo_sa_sum = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum',
    sum_order=sum_order,
    access_type='single'
)

print("=== Single Access - SUM Order ===")
k=3
result = algo_sa_sum.get_k(k)
print(f"k={k}:")
pd.DataFrame([result], columns=['__sum'] + free_variables)

---

#### Query SUM_2

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [SUM_1](#Query-SUM_1)    | 1          | a + b                 | a, b, c        | Y             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('b', 'c'))
]
data = load_dataset("datasets/data_1", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c']
sum_order = {'a': 1, 'b': 1}

# Query SUM_2: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_sum(atoms, data, sum_order, free_variables, output_variables=None)

In [ ]:
# Query SUM_1: Direct Access with SUM order
algo_da_sum = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum', 
    sum_order=sum_order, 
    access_type='direct'
)

print("=== Direct Access - SUM Order ===")
print(f"Total count: {algo_da_sum.get_total_count()}")
print(f"\nFirst result (k=0): {algo_da_sum.get_k(0)}")
print(f"Second result (k=1): {algo_da_sum.get_k(1)}")
print(f"Third result (k=2): {algo_da_sum.get_k(2)}")

print("\nAll results:")
all_results = algo_da_sum.get_all()
pd.DataFrame(all_results, columns=['__sum'] + free_variables)

In [ ]:
# Query SUM_1: Single Access with SUM order
algo_sa_sum = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum',
    sum_order=sum_order,
    access_type='single'
)

print("=== Single Access - SUM Order ===")
k=3
result = algo_sa_sum.get_k(k)
print(f"k={k}:")
pd.DataFrame([result], columns=['__sum'] + free_variables)

---

#### Query SUM_3

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [SUM_3](#Query-SUM_3)    | 2          | a + 2b + c                | a, b, c        | N             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('b', 'c'))
]
data = load_dataset("datasets/data_2", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c']
sum_order = {'a': 1, 'b': 2, 'c': 1}

# Query SUM_3: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_sum(atoms, data, sum_order, free_variables, output_variables=None)

In [ ]:
# Query SUM_3: Direct Access with SUM order
algo_da_sum = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum', 
    sum_order=sum_order, 
    access_type='direct'
)

In [ ]:
# Query SUM_3: Single Access with SUM order
algo_sa_sum = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum',
    sum_order=sum_order,
    access_type='single'
)

print("=== Single Access - SUM Order ===")
k=6
result = algo_sa_sum.get_k(k)
print(f"k={k}:")
pd.DataFrame([result], columns=['__sum'] + free_variables)

---

#### Query SUM_4

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [SUM_4](#Query-SUM_4)    | 3          | a + b + c                     | a, b, c, d        | N             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('b', 'c')),
    ('T', ('c', 'd'))
]
data = load_dataset("datasets/data_3", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c','d']
sum_order = {'a': 1, 'b': 1, 'c': 1}

# Query SUM_4: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_sum(atoms, data, sum_order, free_variables, output_variables=None)

In [ ]:
# Query SUM_4: Direct Access with SUM order
algo_da_sum = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum', 
    sum_order=sum_order, 
    access_type='direct'
)

print("=== Direct Access - SUM Order ===")
print(f"Total count: {algo_da_sum.get_total_count()}")
print(f"\nFirst result (k=0): {algo_da_sum.get_k(0)}")
print(f"Second result (k=1): {algo_da_sum.get_k(1)}")
print(f"Third result (k=2): {algo_da_sum.get_k(2)}")

print("\nAll results:")
all_results = algo_da_sum.get_all()
pd.DataFrame(all_results, columns=['__sum'] + free_variables)

In [ ]:
# Query SUM_4: Single Access with SUM order
algo_sa_sum = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum',
    sum_order=sum_order,
    access_type='single'
)

print("=== Single Access - SUM Order ===")
k=3
result = algo_sa_sum.get_k(k)
print(f"k={k}:")
pd.DataFrame([result], columns=['__sum'] + free_variables)

---

#### Query SUM_5

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [SUM_5](#Query-SUM_5)    | 3          | a + d                     | a, b, c, d     | N             | N             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('b', 'c')),
    ('T', ('c', 'd'))
]
data = load_dataset("datasets/data_3", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c','d']
sum_order = {'a': 1, 'd': 1, }

# Query SUM_5: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_sum(atoms, data, sum_order, free_variables, output_variables=None)

In [ ]:
# Query SUM_5: Direct Access with SUM order
algo_da_sum = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum', 
    sum_order=sum_order, 
    access_type='direct'
)

In [ ]:
# Query SUM_5: Single Access with SUM order --not available
algo_sa_sum = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum',
    sum_order=sum_order,
    access_type='single'
)


---

#### Query SUM_6

| Query ID | Dataset ID | Order (default: ascending)              | Free Variables | Direct Access | Single Access |
|----------|------------|-----------------------------------------|----------------|---------------|---------------|
| [SUM_6](#Query-SUM_6)    | 4          | b + d       | a, b, c, d, e     | Y             | Y             |

Load data

In [ ]:
atoms = [
    ('R', ('a', 'b')),
    ('S', ('a', 'c')),
    ('T', ('a', 'd')),
    ('U', ('a', 'e')),
]
data = load_dataset("datasets/data_4", atoms)

Provide query information and take a look at the expected materialized join result. 

In [ ]:
free_variables = ['a','b','c', 'd', 'e']
sum_order = {'b': 1, 'd': 1}

# Query SUM_6: Use pandas to show join results for reference
print("Expected Join Result:")
pandas_join_sum(atoms, data, sum_order, free_variables, output_variables=None)

In [ ]:
# Query SUM_6: Direct Access with SUM order
algo_da_sum = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum', 
    sum_order=sum_order, 
    access_type='direct'
)

In [ ]:
# Query SUM_6: Single Access with SUM order
algo_sa_sum = AccessAlgorithm(
    atoms=atoms, 
    data=data, 
    free_variables=free_variables,
    order_type='sum',
    sum_order=sum_order,
    access_type='single'
)

print("=== Single Access - SUM Order ===")
k=6
result = algo_sa_sum.get_k(k)
print(f"k={k}:")
pd.DataFrame([result], columns=['__sum'] + free_variables)